# 3 유한 마르코프 결정 과정 (Finite Markov Decision Processes)

## 3.1 에이전트-환경 인터페이스

**유한 마르코프 결정 과정(finite Markov decision process, finite MDP)**은 강화학습 문제를 수학적으로 기술하는 표준 틀이다. MDP에서 에이전트와 환경은 이산 시간 단계 $t = 0, 1, 2, \ldots$마다 상호작용한다. 매 단계에서 에이전트는 현재 상태 $S_t$를 관찰하고 행동 $A_t$를 선택하며, 환경은 보상 $R_{t+1}$과 다음 상태 $S_{t+1}$을 돌려준다.

이 상호작용의 확률적 구조는 다음 함수 $p$로 정의한다.

$$p(s', r \mid s, a) \doteq \Pr\{S_t = s', R_t = r \mid S_{t-1} = s, A_{t-1} = a\}$$

이 틀의 핵심 가정은 **마르코프 성질(Markov property)**이다. 마르코프 성질이란, 현재 상태 $s$가 앞으로의 결정에 필요한 모든 정보를 이미 담고 있다는 것이다. 이 가정이 성립하면, 다음 상태와 보상은 오로지 지금의 $s$와 $a$에만 달려 있고 그 이전에 어떤 경로를 거쳐 왔는지는 관계없다. 따라서 MDP에서 상태란 단순히 지금 눈에 보이는 관측값이 아니라, 과거의 이력을 결정에 필요한 만큼 담아 둔 요약이다.

에이전트와 환경 사이의 경계는 고정된 것이 아니다. 같은 운전 문제를 두고도 행동을 조향각으로 정의할 수도, 목적지 선택으로 정의할 수도 있다. 어느 수준에서 경계를 그을지는 에이전트가 실제로 직접 제어할 수 있는 것이 무엇인지, 행동 한 번이 얼마나 긴 시간 간격을 다루는지에 따라 달라진다. 좋은 경계는 의사결정에 필요한 정보를 빠뜨리지 않으면서, 상태와 행동 공간이 학습 가능한 수준을 벗어나지 않도록 한다.

### Exercise 3.1: MDP를 적용할 수 있는 세 가지 예

마르코프 결정 과정(MDP)은 에이전트가 매 시점 상태를 관찰하고 행동을 선택하면, 보상을 받은 뒤 환경에 따라 다음 상태로 이동하는 상호작용을 기술하는 틀이다. 아래에서는 MDP로 모형화할 수 있는 세 가지 예를 살펴본다.

#### 1. 창고 로봇 내비게이션

자율 로봇이 창고를 돌아다니며 선반에서 물건을 집어 포장 작업장으로 운반하는 과제이다.

- **상태:** 로봇의 격자 위치, 배터리 잔량, 적재 여부, 요청된 물건의 위치, 주변 장애물과 작업자의 위치, 미처리 주문 목록 등.
- **행동:** 동·서·남·북 이동, 정지, 물건 집기, 물건 내려놓기, 충전소로 이동 등.
- **보상:** 배달 성공 시 양의 보상, 매 단계마다 효율성 유도를 위한 작은 음의 보상, 충돌이나 위험 접근 시 큰 음의 보상, 배터리 방전 시 음의 보상.

상태가 물리적으로 명확하고 행동이 구체적이며 보상이 과제 수행 성능과 직결되므로, 이는 전형적인 MDP 사례다.

#### 2. 개인 맞춤형 치료 계획

에이전트가 환자의 상태를 따라가며 적절한 처치를 추천하는 과제이다. 만성 질환 환자의 약물 용량 조절이 한 예다.

- **상태:** 최근 검사 결과, 증상, 나이, 체중, 현재 복용 약물, 치료 이력, 부작용, 주요 위험 요인 등. 환자의 생리적 상태를 완전히 관찰할 수는 없으므로 부분 관측에 가깝다.
- **행동:** 약물 용량 증량·감량, 현재 치료 유지, 약물 변경, 추가 검사 지시, 추적 진료 예약 등.
- **보상:** 건강 지표 개선, 증상 완화, 부작용 감소, 치료 비용 절감, 입원 회피 등을 반영하며, 해로운 결과에는 큰 음의 보상을 부여한다.

이 사례는 환경이 생물학적이고 불확실하며 윤리적 제약이 크다는 점에서 창고 로봇 문제와 성격이 다르다. 또한 건강·안전·비용·삶의 질이라는 이질적 목표를 하나의 스칼라 보상으로 환산하는 일 자체가 어렵고, 그 과정에 가치 판단이 끼어든다.

#### 3. 대화형 이야기 생성 도우미

AI 도우미가 사용자와 함께 이야기를 만들어 나가며, 시점마다 이야기 전개 방향이나 사용자에게 요청할 입력을 결정하는 과제이다.

- **상태:** 지금까지 전개된 이야기, 장르, 등장인물, 미해결 플롯, 사용자의 과거 선호, 현재의 감정적 분위기, 사용자의 몰입도 추정치 등. 사용자의 상상이나 선호를 완전히 관찰할 수 없으므로, 이는 객관적 물리 상태가 아니라 모델이 구성한 표현에 가깝다.
- **행동:** 새로운 사건 추가, 인물 발전, 사용자에게 질문, 갈등 해소, 새로운 배경 도입, 분위기 전환, 이야기 마무리 등.
- **보상:** 사용자 피드백, 지속적 참여, 이야기의 일관성·참신함·감정적 효과, 사용자 제약의 준수 등을 반영하며, 모순된 전개나 지루한 서술, 원치 않는 분위기 전환에는 음의 보상을 부여할 수 있다.

이 예는 MDP 틀이 어디까지 통하는지를 가늠해 볼 수 있는 사례이다. 상태가 객관적이지 않고 보상도 주관적이어서 정의가 까다로우며, 최선의 다음 행동이 대화 전체의 미묘한 맥락에 의존하므로 마르코프 성질이 엄밀히 성립한다고 보기 어렵다. 다만 대화 이력이나 그 임베딩을 상태에 충분히 담으면 마르코프 성질을 근사적으로 회복할 수 있고, 이를 근사로 받아들이면 MDP로 다룰 수 있다.

#### 요약

세 예시는 MDP 틀이 성격이 다른 문제에도 폭넓게 통한다는 점을 보여 준다. 창고 로봇은 구체적인 물리 제어 문제, 치료 계획은 생물학적 불확실성 속에서 신중한 결정을 내려야 하는 의료 문제, 이야기 생성 도우미는 상태와 보상의 정의가 까다로운 주관적 상호작용 문제이다.

### Exercise 3.2: MDP 틀의 적절성

MDP 틀은 추상화 수준이 높아서 여러 목표 지향적 과제를 한데 담아낼 수 있다. 에이전트가 환경과 거듭 상호작용하면서 행동을 고르고 보상을 받아 미래의 결과를 개선해 나가는 구조라면, 그 과제는 대체로 MDP로 기술할 수 있다.

그러나 MDP 표현이 늘 자연스럽거나 실용적이지는 않다. MDP로 표현하려면 상태가 미래 보상과 다음 상태를 예측하는 데 필요한 모든 정보를 담고 있어야 하며, 그렇지 않으면 마르코프 성질이 성립하지 않는다. 다음과 같은 경우에 그 한계가 분명해진다.

**부분 관찰 과제.** 센서가 벽 뒤를 보지 못하는 로봇을 떠올려 보자. 최선 행동이 현재 카메라 영상뿐 아니라 몇 초 전 관측에도 의존한다면, 세계 자체는 마르코프적이더라도 로봇이 받는 관찰은 그렇지 않다. 이런 경우에는 부분 관찰 마르코프 결정 과정(partially observable MDP, POMDP)를 사용하거나, 기억 또는 숨은 상태에 대한 믿음(belief)까지 상태 표현에 담아야 한다.

**보상이 불분명하거나 시간에 따라 변하는 과제.** 사람이 기술을 익히는 동안 자신의 목표나 선호, 무엇을 성공으로 볼지 자체가 함께 변할 수 있다. 현재의 선호를 상태에 넣어 MDP에 담을 수는 있지만, 모델이 부자연스러워지고 다루기 어려워진다.

**비정적 환경.** 과제 규칙, 가능한 행동, 보상의 의미가 변하고 있는데 그 변화가 상태에 들어와 있지 않다면, 표준 MDP 틀만으로는 문제를 표현하기 어렵다.

요컨대 MDP 틀은 모든 목표 지향적 학습 문제를 완벽히 기술하는 모델이라기보다 강력한 추상화 도구로 보는 편이 낫다. 상태를 충분히 넓게 잡으면 많은 문제에 적용할 수 있지만, 부분 관찰성, 숨겨진 과거 의존성, 변화하는 목표, 비정적 동역학이 있을 때는 한계가 또렷해진다.

### Exercise 3.3: 에이전트와 환경의 경계

에이전트와 환경 사이의 경계를 어느 수준에서 설정해야 한다는 유일한 정답은 없다. 경계의 선택은 모델링의 목적, 쓸 수 있는 관찰과 제어, 학습이 일어나야 할 수준에 따라 달라진다. 운전 예에서 행동을 가속 페달, 브레이크, 조향 명령으로 정의하면 인간 운전자의 제어장치 및 일반적인 운전 시스템 인터페이스와 맞아떨어지므로 자연스럽다. 반면 근육 활성화 수준은 운동 제어를 연구할 때만 의미가 있고, 타이어 토크 수준은 자율주행 제어기처럼 그러한 수준의 행동을 직접 다룰 때만 적절하며, "회사로 운전해서 가기" 같은 고수준 선택은 저수준 제어를 따로 다루는 경로 계획 문제에 알맞다.

따라서 경계 선택은 주로 실용적 기준을 따른다. 행동은 에이전트가 실제로 고를 수 있는 결정과 짝지어야 하고, 상태와 행동 표현은 중요한 정보를 빠뜨리지 않으면서 학습 문제를 가능한 한 단순하게 만들어야 하며, 문제의 시간 척도(time scale)와도 맞아야 한다. 어떤 경계가 늘 최선이라고 단정할 원리적 근거는 없으며 같은 시스템을 여러 경계로 기술할 수도 있지만, 잘못된 경계는 문제를 불필요하게 어렵게 만들거나 중요한 인과 구조를 가릴 수 있으므로 선택이 임의적인 것은 아니다. 가장 좋은 경계는 연구 대상 과제에 쓸모 있으면서도 너무 복잡하지 않은 추상화를 주는 경계이다.

<a id="exercise-3-4"></a>

### Exercise 3.4: $p(s', r \mid s, a)$에 대한 표<a id="exercise-3-4"></a>

재활용 로봇 예제에서 상태는 $\text{high}$와 $\text{low}$ 두 가지이고, 가능한 행동은 $\text{search}$, $\text{wait}$, 그리고 $\text{low}$ 상태에서만 가능한 $\text{recharge}$이다. 매개변수 $\alpha$와 $\beta$는 전이 확률을, $r_{\text{search}}$와 $r_{\text{wait}}$는 각각 탐색과 대기 행동의 보상을 나타내며, 배터리가 완전히 방전되어 로봇을 구조해야 할 때는 $-3$의 보상을 받는다.

$p(s', r \mid s, a)$를 정리하면 다음과 같다.

| $s$ | $a$ | $s'$ | $r$ | $p(s', r \mid s, a)$ |
| --- | --- | --- | ---: | ---: |
| $\text{high}$ | $\text{search}$ | $\text{high}$ | $r_{\text{search}}$ | $\alpha$ |
| $\text{high}$ | $\text{search}$ | $\text{low}$ | $r_{\text{search}}$ | $1 - \alpha$ |
| $\text{high}$ | $\text{wait}$ | $\text{high}$ | $r_{\text{wait}}$ | $1$ |
| $\text{low}$ | $\text{search}$ | $\text{low}$ | $r_{\text{search}}$ | $\beta$ |
| $\text{low}$ | $\text{search}$ | $\text{high}$ | $-3$ | $1 - \beta$ |
| $\text{low}$ | $\text{wait}$ | $\text{low}$ | $r_{\text{wait}}$ | $1$ |
| $\text{low}$ | $\text{recharge}$ | $\text{high}$ | $0$ | $1$ |

표에 나타나지 않은 4-튜플의 확률은 모두 $0$이다. 예를 들어 $(\text{high}, \text{wait}, \text{low}, r)$에 해당하는 행이 없는 까닭은, 배터리가 $\text{high}$인 상태에서 $\text{wait}$를 선택하면 $1$의 확률로 그대로 $\text{high}$에 머물기 때문이다. 또 $(\text{high}, \text{recharge}, s', r)$에 해당하는 행이 없는 까닭은, 이 예제에서 $\text{high}$ 상태에서는 $\text{recharge}$가 가능한 행동이 아니기 때문이다.

## 3.2 목표와 보상

강화학습에서는 보상 신호로 목표를 형식화한다. 에이전트의 목적은 사람이 의도한 바를 직접 이해하는 것이 아니라, 시간에 걸쳐 받는 **보상의 누적량을 최대화**하는 것이다. 따라서 보상 설계는 강화학습 문제를 구성할 때 가장 중요한 결정 중 하나이다.

보상은 에이전트가 달성해야 할 **무엇**을 정의해야 하며, 그것을 달성하는 **방법**을 직접 지시해서는 안 된다. 예를 들어 체스 에이전트에는 말을 잡을 때마다 보상을 주기보다 승리에는 양의 보상, 패배에는 음의 보상을 주는 편이 목표에 더 충실하다. 중간 행동을 과도하게 보상하면 에이전트가 진짜 목표가 아니라 설계자가 의도하지 않은 대리 목표를 최적화할 수 있다.

이 절의 요지는 보상 신호가 강화학습 문제에서 목표의 유일한 공식적 표현이라는 데 있다. 좋은 보상은 원하는 결과를 정확히 반영해야 하며, 즉각적인 편의나 직관적인 중간 지표를 보상으로 삼을 때는 그것이 장기 목표와 어긋나지 않는지 주의해야 한다.

### Exercise 3.7: 미로 탈출 로봇의 보상 설계 문제

이 설정의 핵심 문제는 **보상 신호가 학습에 충분한 정보를 제공하지 못한다**는 점이다. 탈출 시 $+1$, 그 외에는 모두 $0$인 종단 보상은 다음과 같은 두 가지 어려움을 낳는다.

첫째, 에피소딕 과제이고 할인이 없다면(또는 $\gamma = 1$이라면) 모든 성공 에피소드의 이득은 똑같이 $1$이다. 빨리 탈출하든 오래 헤매다 탈출하든 이득에 차이가 없으므로, 에이전트는 탈출을 서두를 이유가 없다. 즉 보상이 "탈출하라"는 의도는 전달하지만 "**빨리** 탈출하라"는 의도는 전달하지 못한다.

둘째, 초기 정책이 무작위에 가까울 때 탈출에 도달하는 일이 매우 드물다면 양의 보상 신호 자체를 거의 받지 못하게 되어 학습이 진척되지 않는다. 즉 보상이 너무 희소(sparse)하다.

따라서 의도가 제대로 전달되었다고 보기 어렵다. 이를 개선하려면 매 단계마다 작은 음의 보상($-1$ 등)을 주어 빠른 탈출을 유도하거나, 할인 인자 $\gamma < 1$을 도입해 늦은 성공일수록 이득이 작아지도록 만들면 된다. 두 방법 모두 "가능한 한 빨리 탈출하라"는 목표를 보상 신호에 명시적으로 반영한다.

## 3.3 이득과 에피소드

에이전트가 최대화하려는 양은 단일 보상이 아니라 시간에 따른 보상의 누적값인 **이득(return)**이다. 시점 $t$ 이후에 받을 보상 수열 $R_{t+1}, R_{t+2}, \ldots$에 대해, 가장 단순한 에피소딕 과제의 이득은 다음과 같이 쓴다.

$$G_t \doteq R_{t+1} + R_{t+2} + \cdots + R_T$$

여기서 $T$는 에피소드가 끝나는 시점이다. 에피소딕 과제(episodic task)는 게임 한 판이나 미로 탈출처럼 자연스러운 종료 시점이 있는 문제이며, 종료 후에는 특별한 **종단 상태(terminal state)**에 도달한다고 볼 수 있다.

반대로 공장 제어, 로봇 운용, 온라인 서비스 운영처럼 명확한 종료 시점이 없는 문제는 **연속 과제(continuing task)**로 다룬다. 이 경우 단순 합은 무한히 커질 수 있으므로, 미래 보상에 할인 인자 $\gamma \in [0, 1]$를 곱해 다음과 같은 할인 이득을 사용한다.

$$G_t \doteq R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \cdots = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}$$

$\gamma$가 작을수록 에이전트는 가까운 미래를 더 중시하고, $\gamma$가 $1$에 가까울수록 먼 미래의 보상까지 중요하게 고려한다. 이득은 재귀적으로 $G_t = R_{t+1} + \gamma G_{t+1}$이라고 쓸 수 있으며, 이후 벨만 방정식은 이 단순한 관계를 기초로 삼는다.

### Exercise 3.5: 식 (3.3)의 에피소딕 형태 수정

3.1절의 식 (3.3)은 연속(continuing) 과제에 대한 것으로,

$$
\sum_{s' \in \mathcal{S}} \sum_{r \in \mathcal{R}} p(s', r \mid s, a) = 1,
\qquad \text{모든 } s \in \mathcal{S},\ a \in \mathcal{A}(s) \text{에 대해}
$$

와 같이 쓸 수 있다. 즉 어떤 상태 $s$에서 행동 $a$를 취했을 때 가능한 모든 다음 상태와 보상에 대한 확률의 합이 $1$임을 뜻한다.

에피소딕 과제에서는 상태 집합에 **종단 상태(terminal state)** 를 포함시킨 확장 집합 $\mathcal{S}^+ = \mathcal{S} \cup \{\text{종단 상태}\}$를 사용한다. 비종단 상태에서 전이하면 종단 상태로 이동할 가능성이 있고, 이 경우에도 확률의 총합은 $1$이 되어야 하므로, 다음 상태 $s'$에 대한 합의 범위를 $\mathcal{S}$에서 $\mathcal{S}^+$로 확장해야 한다. 따라서 식 (3.3)의 에피소딕 형태는 다음과 같다.

$$
\sum_{s' \in \mathcal{S}^+} \sum_{r \in \mathcal{R}} p(s', r \mid s, a) = 1,
\qquad \text{모든 } s \in \mathcal{S},\ a \in \mathcal{A}(s) \text{에 대해.}
$$

여기서 행동의 출발 상태 $s$는 비종단 상태로 한정하므로 여전히 $\mathcal{S}$에 속한다(종단 상태에서는 더 이상 행동을 고르지 않는다). 다만 다음 상태 $s'$는 종단 상태일 수 있으므로 $\mathcal{S}^+$ 위에서 합한다는 점이 핵심적인 차이이다.

### Exercise 3.6: 에피소딕 + 할인 형태의 막대 균형 잡기

이 문제에서는 막대 균형 잡기를 에피소딕 과제로 다루되 할인을 함께 사용하며, 실패 시 보상이 $-1$이고 그 외 모든 시점의 보상은 $0$이라고 가정한다. 한 에피소드 안에서 시점 $T$에 균형이 무너졌다고 할 때, $R_T = -1$이고 그 밖의 모든 보상은 $0$이다.

시점 $t$($t < T$)에서의 이득은

$$
G_t = \sum_{k=0}^{T-t-1} \gamma^k R_{t+k+1} = -\gamma^{T-t-1}
$$

이 된다. 따라서 실패까지 남은 시간이 길수록 절댓값이 작아지고($\gamma < 1$), 실패 직전에는 $G_{T-1} = -1$로 가장 큰 벌을 받는다.

연속 형태(continuing formulation)에서는 실패가 에피소드를 종료시키지 않으므로, 미래의 모든 실패 시점 $T_1 < T_2 < \cdots$에 대해

$$
G_t = -\sum_{i:\, T_i > t} \gamma^{T_i - t - 1}
$$

이 된다. 즉 에피소딕 형태에서는 한 번의 실패까지만 이득에 반영되어 단일 항이 되는 반면, 연속 형태에서는 미래의 모든 실패가 쌓이므로 일반적으로 절댓값이 더 크다. 다만 두 형태 모두 "실패를 가능한 한 늦추는 것"이 이득을 최대화하는 행동이라는 점은 같으며, 정책이 학습되어 가는 방향도 동일하다.

### Exercise 3.8: 유한 보상 수열의 이득 계산

$\gamma = 0.5$이고 $R_1 = -1$, $R_2 = 2$, $R_3 = 6$, $R_4 = 3$, $R_5 = 2$, $T = 5$일 때 $G_0, G_1, \ldots, G_5$를 구한다. 종단 이후 보상이 없으므로 $G_5 = 0$이며, $G_t = R_{t+1} + \gamma\, G_{t+1}$을 이용해 거꾸로 계산하면 다음과 같다.

$$
\begin{aligned}
G_5 &= 0 \\
G_4 &= R_5 + 0.5 \cdot G_5 = 2 + 0 = 2 \\
G_3 &= R_4 + 0.5 \cdot G_4 = 3 + 0.5 \cdot 2 = 4 \\
G_2 &= R_3 + 0.5 \cdot G_3 = 6 + 0.5 \cdot 4 = 8 \\
G_1 &= R_2 + 0.5 \cdot G_2 = 2 + 0.5 \cdot 8 = 6 \\
G_0 &= R_1 + 0.5 \cdot G_1 = -1 + 0.5 \cdot 6 = 2
\end{aligned}
$$

따라서 $(G_0, G_1, G_2, G_3, G_4, G_5) = (2,\, 6,\, 8,\, 4,\, 2,\, 0)$이다.

### Exercise 3.9: 무한 보상 수열의 이득 계산

$\gamma = 0.9$이고 $R_1 = 2$ 이후로는 $7$이 무한히 이어진다고 하자.

$G_1$은 $R_2$부터 시작하는 모든 $7$의 할인된 합이므로 무한 등비급수가 된다.

$$
G_1 = \sum_{k=0}^{\infty} \gamma^k \cdot 7 = \frac{7}{1 - 0.9} = 70.
$$

$G_0$은 첫 보상 $R_1 = 2$를 더한 뒤 $G_1$을 한 번 할인한 값이다.

$$
G_0 = R_1 + \gamma\, G_1 = 2 + 0.9 \cdot 70 = 2 + 63 = 65.
$$

따라서 $G_0 = 65$, $G_1 = 70$이다.

### Exercise 3.10: 식 (3.10)의 두 번째 등식 증명

식 (3.10)은 다음과 같다.

$$
G_t = \sum_{k=0}^{\infty} \gamma^k = \frac{1}{1 - \gamma}.
$$

증명할 등식은 $\sum_{k=0}^{\infty} \gamma^k = \dfrac{1}{1 - \gamma}$이며, 이는 $0 \le \gamma < 1$일 때 성립한다.

부분합을 $S_n = \sum_{k=0}^{n} \gamma^k$로 두고 양변에 $\gamma$를 곱하면

$$
\gamma S_n = \sum_{k=0}^{n} \gamma^{k+1} = \sum_{k=1}^{n+1} \gamma^k.
$$

두 식을 빼면 가운데 항들이 모두 소거되어

$$
S_n - \gamma S_n = 1 - \gamma^{n+1}, \qquad \text{즉}\quad S_n = \frac{1 - \gamma^{n+1}}{1 - \gamma}
$$

가 된다. $0 \le \gamma < 1$이면 $n \to \infty$일 때 $\gamma^{n+1} \to 0$이므로

$$
\sum_{k=0}^{\infty} \gamma^k = \lim_{n \to \infty} S_n = \frac{1}{1 - \gamma}.
$$

따라서 두 번째 등식이 성립한다. $\blacksquare$

## 3.4 에피소딕 과제와 연속 과제의 통합 표기

에피소딕 과제와 연속 과제는 겉으로는 다르지만, 이후 알고리즘을 전개할 때는 하나의 표기로 다루는 것이 편리하다. 이를 위해 종단 상태를 포함한 상태 집합 $\mathcal{S}^+$를 사용하고, 에피소드가 끝난 뒤에는 더 이상 보상이 누적되지 않는 것으로 본다.

에피소딕 과제에서는 $t = 0, 1, \ldots, T-1$까지만 실제 행동을 선택하며, $S_T$를 종단 상태로 둔다. 연속 과제에서는 $T = \infty$라고 생각하면 충분하다. 이렇게 보면 이득은 다음과 같은 하나의 식으로 쓸 수 있다.

$$G_t \doteq \sum_{k=t+1}^{T} \gamma^{k-t-1} R_k$$

이 표기는 에피소딕 과제에서 $\gamma = 1$을 허용하면서도, 연속 과제에서는 보통 $\gamma < 1$을 두어 무한 합이 잘 정의되게 한다. 중요한 점은 과제의 형식보다 에이전트가 장기적으로 어떤 보상 합을 최적화하는지가 중심이라는 것이다.

<a id="example-3-5"></a>


## 3.5 정책과 가치 함수<a id="example-3-5"></a>

**정책(policy)** $\pi$는 각 상태에서 어떤 행동을 선택할지를 나타내는 규칙이다. 확률적 정책은 $\pi(a \mid s)$로 쓰며, 이는 상태 $s$에서 행동 $a$를 선택할 확률을 뜻한다. 강화학습의 목표는 좋은 정책을 찾는 것이고, 우리는 그 정책을 따랐을 때의 기대 이득으로 정책의 좋고 나쁨을 평가한다.

상태가치함수는 상태 $s$에서 시작해 정책 $\pi$를 따를 때의 기대 이득이다.

$$v_\pi(s) \doteq \mathbb{E}_\pi[G_t \mid S_t = s]$$

행동가치함수는 상태 $s$에서 먼저 행동 $a$를 취한 뒤, 이후 정책 $\pi$를 따를 때의 기대 이득이다.

$$q_\pi(s, a) \doteq \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a]$$

이 가치 함수들은 현재 상태나 행동의 장기적 가치를 수치로 나타낸다. 즉 어떤 상태가 좋은지, 어떤 행동이 좋은지를 판단할 때는 즉시 보상만 보지 않고, 그 뒤에 이어질 상태들과 보상까지 함께 고려한다. 이 점이 강화학습을 단순한 즉시 보상 최대화 문제와 구분한다.

이득의 재귀식과 MDP 동역학을 결합하면 $v_\pi$에 대한 **벨만 방정식(Bellman equation)**이 나온다.

$$v_\pi(s) = \sum_a \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\bigl[r + \gamma v_\pi(s')\bigr]$$

이 식은 가능한 다음 상태들의 가치와 즉시 보상의 가중평균이 한 상태의 가치를 정함을 뜻한다. 뒤 장들에서 등장하는 동적 계획법, 몬테카를로 방법, 시간 차분 학습은 모두 이 관계를 직접 또는 간접적으로 이용한다.

### Exercise 3.11: 정책 $\pi$ 하에서 $R_{t+1}$의 기댓값

현재 상태가 $S_t = s$이고 행동이 확률적 정책 $\pi(a \mid s)$에 따라 선택될 때, 한 단계 보상 $R_{t+1}$의 기댓값을 정책 $\pi$와 4-인자 함수 $p(s', r \mid s, a)$로 표현하고자 한다.

보상 $R_{t+1}$은 두 단계의 확률적 과정을 거쳐 정해진다. 먼저 상태 $s$에서 정책 $\pi(a \mid s)$가 행동 $a$를 뽑고, 이어서 환경의 동역학 $p(s', r \mid s, a)$가 다음 상태 $s'$와 보상 $r$을 함께 정한다. 따라서 기댓값은 가능한 모든 행동 $a$, 다음 상태 $s'$, 보상 $r$에 대한 합으로 쓸 수 있다.

$$
\mathbb{E}[R_{t+1} \mid S_t = s] = \sum_{a \in \mathcal{A}(s)} \pi(a \mid s) \sum_{s' \in \mathcal{S}} \sum_{r \in \mathcal{R}} r \cdot p(s', r \mid s, a).
$$

이 식은 두 단계로 평균을 취하는 구조다. 안쪽의 이중 합 $\sum_{s', r} r \cdot p(s', r \mid s, a)$는 상태 $s$에서 행동 $a$를 취했을 때의 기대 보상 $r(s, a)$에 해당하며, 바깥의 합은 이 행동별 기대 보상을 정책 확률 $\pi(a \mid s)$로 가중평균한다. 즉 "행동을 어떻게 고를지"와 "그 행동의 결과가 어떻게 나올지"라는 두 단계의 불확실성을 차례로 평균낸다.

<a id="exercise-3-12"></a>

### Exercise 3.12: $v_\pi$를 $q_\pi$와 $\pi$로 표현<a id="exercise-3-12"></a>

상태가치함수 $v_\pi(s)$는 상태 $s$에서 출발하여 정책 $\pi$를 따랐을 때의 기대 이득이고, 행동가치함수 $q_\pi(s, a)$는 상태 $s$에서 행동 $a$를 취한 뒤 이후 정책 $\pi$를 따랐을 때의 기대 이득이다. 두 함수의 정의를 비교하면, $v_\pi(s)$는 첫 행동을 정책 $\pi$에 따라 선택한 경우의 $q_\pi(s, a)$를 모든 행동에 대해 평균낸 것에 해당한다.

따라서 다음 관계가 성립한다.

$$
v_\pi(s) = \sum_{a \in \mathcal{A}(s)} \pi(a \mid s)\, q_\pi(s, a).
$$

이 식은 상태의 가치가 그 상태에서 가능한 모든 행동의 가치를 정책 확률로 가중평균한 값임을 뜻한다. 정책 $\pi$가 결정적(deterministic)이어서 어떤 행동 $a^*$를 확률 $1$로 선택한다면, 이 합은 단일 항으로 줄어들어 $v_\pi(s) = q_\pi(s, a^*)$가 된다.

<a id="exercise-3-13"></a>

### Exercise 3.13: $q_\pi$를 $v_\pi$와 4-인자 $p$로 표현<a id="exercise-3-13"></a>

행동가치함수 $q_\pi(s, a)$는 상태 $s$에서 행동 $a$를 취했을 때의 기대 이득이며, 이는 한 단계 보상과 그 이후의 할인된 이득으로 쪼갤 수 있다. 즉

$$
q_\pi(s, a) = \mathbb{E}[R_{t+1} + \gamma\, G_{t+1} \mid S_t = s, A_t = a]
$$

이다. 상태 $s$에서 행동 $a$를 취하면 분포 $p(s', r \mid s, a)$가 다음 상태 $s'$와 보상 $r$을 정하고, 다음 상태 $s'$에서부터는 정책 $\pi$를 따르므로 그 이후의 기대 이득은 정의에 의해 $v_\pi(s')$와 같다. 이를 정리하면 다음과 같다.

$$
q_\pi(s, a) = \sum_{s' \in \mathcal{S}} \sum_{r \in \mathcal{R}} p(s', r \mid s, a)\,\bigl[r + \gamma\, v_\pi(s')\bigr].
$$

이 식은 "행동의 가치는 즉시 받는 보상과 할인된 다음 상태의 가치를 합한 양의 기댓값"이라는 직관을 그대로 표현하며, $v_\pi$로부터 $q_\pi$를 한 단계 펼쳐 보는(one-step lookahead) 형태에 해당한다.

[Exercise 3.12](#exercise-3-12)와 [3.13](#exercise-3-13)의 두 식을 결합하면 잘 알려진 $v_\pi$에 대한 벨만 방정식

$$
v_\pi(s) = \sum_{a} \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\,\bigl[r + \gamma\, v_\pi(s')\bigr]
$$

을 자연스럽게 얻는다. 두 연습문제는 항등식 한 줄에 그치는 듯하지만, 사실은 강화학습 전반에서 거듭 등장하는 벨만 방정식의 두 구성 요소를 따로 떼어 놓고 본 셈이다.

### Exercise 3.14: 중앙 상태에 대한 벨만 방정식 수치 검증

[예제 3.5](#example-3-5)의 그림 3.2(오른쪽)에 나타난 가치함수 $v_\pi$는 균등 무작위 정책($\pi(a \mid s) = 0.25$, 네 가지 행동 각각)과 할인율 $\gamma = 0.9$ 아래에서 계산된 값이다. 중앙 상태의 값은 $+0.7$이고, 그 네 이웃(북·동·남·서)의 값은 $+2.3,\ +0.4,\ -0.4,\ +0.7$이다. 중앙 상태에서는 어떤 방향으로 이동하더라도 격자 안에 머물고 특수 상태(A, B)도 아니므로 즉시 보상은 모두 $0$이고, 전이는 결정적이다.

벨만 방정식 (3.14)는 두 개의 합을 포함한다.

$$
v_\pi(s) = \sum_{a} \pi(a \mid s) \sum_{s', r} p(s', r \mid s, a)\,\bigl[r + \gamma\, v_\pi(s')\bigr]
$$

이 식이 중앙 상태에서 어떻게 한 줄짜리 산술식으로 정리되는지를 두 시그마를 차례대로 풀어 가며 살펴본다.

**바깥쪽 합 $\sum_a \pi(a \mid s)$의 전개.** 가능한 행동은 네 가지(북·동·남·서)이고 정책이 균등 무작위이므로 $\pi(a \mid s) = 0.25$이다. 따라서 바깥쪽 합은 네 개의 항으로 펼쳐지며, 각 항의 가중치는 $0.25$이다.

$$
\sum_{a} \pi(a \mid s)\,[\,\cdot\,] = 0.25\,[\,\cdot\,]_{북} + 0.25\,[\,\cdot\,]_{동} + 0.25\,[\,\cdot\,]_{남} + 0.25\,[\,\cdot\,]_{서}.
$$

**안쪽 합 $\sum_{s', r} p(s', r \mid s, a)$의 축약.** 격자 세계는 일반 이동에 대해 결정적이다. 즉 중앙 상태에서 행동 $a$를 취하면 그 방향의 이웃 $s'_a$로 확률 $1$로 이동하고 보상은 항상 $0$이다. 따라서 $p(s'_a, 0 \mid s, a) = 1$이고 그 밖의 모든 $(s', r)$ 조합에 대한 확률은 $0$이므로, 합은 단 하나의 항으로 축약된다.

$$
\sum_{s', r} p(s', r \mid s, a)\,\bigl[r + \gamma\, v_\pi(s')\bigr] = 1 \cdot \bigl[\,0 + \gamma\, v_\pi(s'_a)\,\bigr] = 0.9\, v_\pi(s'_a).
$$

만약 환경이 확률적이었다면(예: 의도한 방향과 다른 곳으로 미끄러질 가능성이 있는 경우) 이 합은 여러 항의 가중합으로 남았을 것이다.

**두 단계를 결합.** 두 결과를 합치면 벨만 방정식은 중앙 상태에서 다음과 같이 정리된다.

$$
v_\pi(s) = 0.25 \sum_{s' \in \text{이웃}} \bigl[0 + 0.9\, v_\pi(s')\bigr] = 0.25 \times 0.9 \times \sum_{s' \in \text{이웃}} v_\pi(s').
$$

네 이웃의 값을 대입하면

$$
v_\pi(s) = 0.25 \times 0.9 \times (2.3 + 0.4 + (-0.4) + 0.7) = 0.25 \times 0.9 \times 3.0 = 0.675
$$

을 얻는다. 본문이 표의 $v_\pi$를 소수 첫째 자리까지만 정확하다고 밝혔으므로 $0.675 \approx 0.7$로 반올림하면 중앙값과 일치한다. 따라서 그림의 숫자를 직접 대입했을 때 벨만 방정식의 좌변과 우변이 (반올림 오차 범위 안에서) 같다는 의미에서 중앙 상태에서 벨만 방정식이 수치적으로 성립한다.

### Exercise 3.15: 모든 보상에 상수를 더하는 효과 (연속 과제)

격자 세계 예제에서 보상의 절대 부호가 본질적인지, 아니면 보상들 사이의 차이만 본질적인지를 묻는 문제이다. 모든 보상에 동일한 상수 $c$를 더해도 정책의 상대적 가치 순위가 바뀌지 않음을 식 (3.8)을 이용해 보인다.

식 (3.8)에 따라 시점 $t$의 이득은

$$
G_t = \sum_{k=0}^{\infty} \gamma^k R_{t+k+1}
$$

이다. 모든 보상에 $c$를 더한 새로운 보상 $R'_{t+k+1} = R_{t+k+1} + c$로 정의된 이득을 $G'_t$라 하면

$$
G'_t = \sum_{k=0}^{\infty} \gamma^k (R_{t+k+1} + c) = G_t + c \sum_{k=0}^{\infty} \gamma^k = G_t + \frac{c}{1 - \gamma}
$$

이 된다. 양변의 기댓값을 구하면 모든 상태 $s$에 대해

$$
v'_\pi(s) = v_\pi(s) + \frac{c}{1 - \gamma}, \qquad v_c = \frac{c}{1 - \gamma}
$$

가 성립한다. 즉 모든 상태의 가치가 같은 상수 $v_c$만큼 평행이동할 뿐이므로 두 정책 $\pi, \pi'$를 비교할 때 $v_\pi(s) - v_{\pi'}(s)$의 값은 변하지 않는다. 따라서 어떤 정책의 우열도 바뀌지 않으며, 연속 과제에서는 보상의 부호가 본질적이지 않고 오직 보상들 사이의 차이만이 의미를 가진다.

### Exercise 3.16: 모든 보상에 상수를 더하는 효과 (에피소딕 과제)

에피소딕 과제에서는 같은 조작이 결과를 바꿀 수 있다. 핵심 차이는 에피소드의 길이가 정책에 따라 달라진다는 점이다. 에피소드의 길이를 $T$라 하면 이득은

$$
G_0 = \sum_{k=0}^{T-1} \gamma^k R_{k+1}
$$

이고, 보상에 $c$를 더하면

$$
G'_0 = G_0 + c \sum_{k=0}^{T-1} \gamma^k = G_0 + c \cdot \frac{1 - \gamma^T}{1 - \gamma}
$$

이 된다. 추가되는 항이 더 이상 상수가 아니라 에피소드 길이 $T$에 의존하므로, 에피소드를 짧게 끝내는 정책과 길게 끄는 정책 사이의 상대적 가치가 달라질 수 있다.

미로 탈출 문제가 좋은 예이다. 원래 보상 설계가 매 단계 $-1$, 탈출 시 $0$이라면 빠르게 탈출할수록 이득이 커지므로 에이전트는 최단 경로를 학습한다. 여기에 $c = +1$을 더해 매 단계 $0$, 탈출 시 $+1$로 바꾸면, 어떤 경로로 탈출하든 이득은 항상 $+1$ 부근이 되어 빠른 탈출의 동기가 사라진다. 더 극단적으로 $c = +2$를 더하면 매 단계 $+1$, 탈출 시 $+2$가 되어 에피소드를 오래 끌수록 이득이 커지므로, 에이전트는 오히려 미로 안에서 가능한 한 오래 머무는 정책을 선호하게 된다.

따라서 에피소딕 과제에서는 모든 보상에 상수를 더하는 일이 과제를 본질적으로 바꿀 수 있고, 보상의 부호와 절대적 크기가 의미를 갖는다.

<a id="exercise-3-17"></a>

### Exercise 3.17: 행동 가치에 대한 벨만 방정식<a id="exercise-3-17"></a>

상태 가치 $v_\pi$에 대한 벨만 방정식 (3.14)와 유사한 식을 행동 가치 $q_\pi$에 대해 유도한다. 정의에 의해

$$
q_\pi(s, a) = \mathbb{E}_\pi[G_t \mid S_t = s, A_t = a] = \mathbb{E}_\pi[R_{t+1} + \gamma\, G_{t+1} \mid S_t = s, A_t = a]
$$

이다. 상태 $s$에서 행동 $a$를 취하면 환경의 동역학 $p(s', r \mid s, a)$가 다음 상태 $s'$와 보상 $r$을 정하고, 이후 상태 $s'$에서는 정책 $\pi$가 다음 행동 $a'$를 뽑으며 그 이후의 기대 이득은 $q_\pi(s', a')$이다. 이를 펼쳐 쓰면 다음과 같다.

$$
q_\pi(s, a) = \sum_{s', r} p(s', r \mid s, a)\Bigl[r + \gamma \sum_{a'} \pi(a' \mid s')\, q_\pi(s', a')\Bigr]
$$

이 식이 곧 $q_\pi$에 대한 벨만 방정식이며, 문제에서 제시된 백업 다이어그램에서 가지 $p$는 환경의 전이를, 가지 $\pi$는 다음 상태에서의 정책에 따른 행동 선택을 나타낸다.

<a id="exercise-3-18"></a>

### Exercise 3.18: $v_\pi(s)$를 $q_\pi(s, a)$로 표현 (정책 평균)<a id="exercise-3-18"></a>

상태 $s$의 가치는 그 상태에서 가능한 각 행동의 가치를, 현재 정책 $\pi$가 그 행동을 선택할 확률로 가중평균한 값이다. 이를 정책에 대한 기댓값 형태로 쓰면

$$
v_\pi(s) = \mathbb{E}_\pi[q_\pi(s, A_t) \mid S_t = s]
$$

이다. 이 기댓값은 행동 $A_t$를 정책 $\pi(\cdot \mid s)$로 뽑는다는 조건에서 잡는다. 기댓값 표기를 풀어 쓰면

$$
v_\pi(s) = \sum_{a} \pi(a \mid s)\, q_\pi(s, a)
$$

가 된다. 백업 다이어그램에서 루트 $s$로부터 각 자식 $a_1, a_2, a_3$로 뻗는 가지의 확률이 $\pi(a \mid s)$이고, 자식 노드의 값이 $q_\pi(s, a)$이므로 두 식 모두 다이어그램과 정확히 대응한다.

<a id="exercise-3-19"></a>

### Exercise 3.19: $q_\pi(s, a)$를 $R_{t+1}$과 $v_\pi(S_{t+1})$로 표현 (환경 평균)<a id="exercise-3-19"></a>

행동 가치는 즉시 받는 보상과 다음 상태에서부터의 기대 이득의 합이며, 다음 상태에서부터는 정책 $\pi$를 따르므로 그 부분의 기대 이득은 $v_\pi(S_{t+1})$이다. 이때 기댓값은 환경의 동역학에 대해서만 잡으며 정책 쪽에는 따로 조건을 두지 않는다(첫 행동 $a$가 이미 고정되어 있기 때문이다). 따라서

$$
q_\pi(s, a) = \mathbb{E}\bigl[R_{t+1} + \gamma\, v_\pi(S_{t+1}) \,\bigm|\, S_t = s,\ A_t = a\bigr]
$$

이고, 기댓값을 4-인자 함수 $p(s', r \mid s, a)$로 풀어 쓰면

$$
q_\pi(s, a) = \sum_{s', r} p(s', r \mid s, a)\,\bigl[r + \gamma\, v_\pi(s')\bigr]
$$

이 된다. 백업 다이어그램에서 루트 $(s, a)$로부터 각 다음 상태 $s'_1, s'_2, s'_3$로 뻗는 가지의 보상이 $r_i$, 가지 확률이 $p(s', r \mid s, a)$, 자식 노드의 값이 $v_\pi(s')$이므로 두 식은 다이어그램의 구조를 그대로 반영한다. 이 식과 [Exercise 3.18](#exercise-3-18)의 식을 합치면 [Exercise 3.17](#exercise-3-17)의 $q_\pi$에 대한 벨만 방정식이 다시 나온다.

## 3.6 최적 정책과 최적 가치 함수

정책 $\pi$가 모든 상태에서 정책 $\pi'$보다 크거나 같은 가치를 주면, $\pi$는 $\pi'$보다 좋거나 같다고 말한다. 유한 MDP에서는 적어도 하나의 **최적 정책(optimal policy)**이 존재하며, 모든 최적 정책은 동일한 최적 상태가치함수 $v_*$를 공유한다.

$$v_*(s) \doteq \max_\pi v_\pi(s)$$

마찬가지로 최적 행동가치함수는 다음과 같이 쓴다.

$$q_*(s, a) \doteq \max_\pi q_\pi(s, a)$$

최적 가치 함수는 벨만 방정식의 최적성 형태인 **벨만 최적 방정식(Bellman optimality equation)**을 만족한다.

$$v_*(s) = \max_a \sum_{s', r} p(s', r \mid s, a)\bigl[r + \gamma v_*(s')\bigr]$$

이 식은 가능한 행동들 중 가장 큰 한 단계 기대 이득이 최적 상태 가치를 결정한다는 뜻이다. 만약 $v_*$를 정확히 알고 있다면, 각 상태에서 위 식의 최댓값을 만드는 행동을 선택하면 최적 정책이 된다. 마찬가지로 $q_*$를 알고 있다면 환경 모델 없이도 단순히 $\arg\max_a q_*(s, a)$를 선택하면 충분하다.

다만 이 결론은 계산적으로는 이상적 기준에 가깝다. 실제 문제에서는 환경의 전이 확률을 모르거나, 상태 공간이 너무 커서 모든 상태의 정확한 최적 가치를 계산할 수 없는 경우가 대부분이다. 이후 장들의 알고리즘은 이러한 이상적 방정식을 경험으로부터 근사적으로 푸는 방법이다.

### Exercise 3.20: 골프 예제의 최적 상태 가치함수

골프 예제에서는 매 타격마다 보상 $-1$을 받고 공이 홀에 들어가면 에피소드가 종료된다. 에이전트는 두 종류의 행동 — 멀리 보내지만 정밀도가 떨어지는 **드라이버**와, 짧지만 정밀한 **퍼터** — 중 하나를 선택할 수 있다. 최적 정책 $\pi_*$는 그린 위에서는 퍼터를, 그 밖의 영역에서는 드라이버를 사용하는 정책이며, $v_*$는 이 정책 아래에서의 기대 타수에 음의 부호를 붙인 값이다.

각 영역의 값은 다음과 같이 묘사된다.

- **그린 위 (홀 근처):** 한 번의 퍼팅으로 홀에 들어가므로 $v_*(s) = -1$.
- **드라이버로 한 번에 그린에 올릴 수 있는 영역:** 드라이버 한 번으로 그린에 올린 뒤 퍼팅 한 번으로 마무리하므로 $v_*(s) = -2$. 이 등치선(contour)은 그린을 둘러싸는 비교적 넓은 영역을 이룬다.
- **그보다 멀어 드라이버 한 번으로는 그린에 닿지 않는 영역:** 드라이버 한 번으로 위의 $-2$ 영역에 진입한 뒤 다시 드라이버 한 번, 마지막으로 퍼팅 한 번이 필요하므로 $v_*(s) = -3$.
- **더 먼 영역:** 동일한 논리로 $-4, -5, \ldots$의 등치선이 차례로 그려진다.

요약하면 $v_*$의 등치선은 홀을 중심으로 동심원 형태로 펼쳐지며, 안쪽이 $-1$, 그 밖이 차례로 $-2, -3, \ldots$이고, 그린 가장자리와 드라이버 사거리 경계에서 등치선이 꺾인다.

### Exercise 3.21: $q_*(s, \texttt{putter})$의 등치선

$q_*(s, \texttt{putter})$는 첫 행동을 강제로 퍼터로 고정하고 그 이후에만 최적 정책을 따랐을 때의 가치이다. 즉

$$
q_*(s, \texttt{putter}) = \mathbb{E}\bigl[R_{t+1} + \gamma\, v_*(S_{t+1}) \,\bigm|\, S_t = s,\ A_t = \texttt{putter}\bigr]
$$

이며, 골프 예제에서는 $\gamma = 1$, $R_{t+1} = -1$이므로 $q_*(s, \texttt{putter}) = -1 + v_*(s')$ 형태가 된다. 여기서 $s'$는 퍼터로 한 번 친 뒤 도달하는 상태이다.

영역별로 보면 다음과 같다.

- **홀 바로 옆 (퍼팅 한 번에 들어가는 영역):** 퍼팅으로 종료되므로 $q_*(s, \texttt{putter}) = -1$.
- **그린 위 그 밖:** 퍼팅 한 번으로 더 가까이 이동하지만 아직 그린 위라 다음 타구도 퍼팅 한 번이면 끝난다. 따라서 $q_*(s, \texttt{putter}) = -2$.
- **그린 바깥, 그러나 퍼터로 그린 위에 올릴 수 있는 영역:** 퍼팅으로 그린에 진입하면 그 위치의 $v_*$는 $-2$이므로 $q_*(s, \texttt{putter}) = -1 + (-2) = -3$.
- **그린 바깥에서 퍼터로 한 번 쳐도 여전히 그린 바깥인 영역:** 퍼터의 사거리가 짧아 한 번에 그린에 못 미치고, 다음 상태도 그린 밖이므로 그곳의 $v_*$는 드라이버를 쓰는 영역의 값(예: $-2, -3, \ldots$)이 된다. 따라서 $q_*(s, \texttt{putter}) = -1 + v_*(s')$가 되어 $-3, -4, \ldots$ 등치선이 더 멀리까지 퍼진다. 다만 퍼터의 사거리에 한계가 있어 어느 거리 이상에서는 첫 타로 사실상 진척이 없으므로 $q_*$는 $-\infty$에 가까워진다.

요약하면 $q_*(s, \texttt{putter})$의 등치선은 $v_*$와 비슷하지만, 첫 타에 드라이버 대신 퍼터를 써야 한다는 제약 때문에 같은 거리에서 한 단계씩 값이 낮고, 등치선이 홀 쪽으로 당겨진다.

### Exercise 3.22: 두 결정적 정책의 비교

문제의 연속 MDP는 윗 상태에서 두 행동 $\texttt{left}, \texttt{right}$ 중 하나를 선택하고, 결정적 보상을 받은 뒤 자식 상태로 이동한 다음, 거기서 다시 결정적으로 윗 상태로 돌아오는 구조이다. 행동 $\texttt{left}$를 선택하면 즉시 보상 $+1$, 자식 상태에서 윗 상태로 돌아올 때 보상 $0$을 받고, 행동 $\texttt{right}$를 선택하면 즉시 보상 $0$, 자식 상태에서 윗 상태로 돌아올 때 보상 $+2$를 받는다. 따라서 두 정책 아래에서 윗 상태에서 출발하는 보상 수열은

$$
\pi_{\texttt{left}}: \ 1, 0, 1, 0, 1, 0, \ldots, \qquad \pi_{\texttt{right}}: \ 0, 2, 0, 2, 0, 2, \ldots
$$

이고, 두 보상이 두 단계마다 주기적으로 반복된다. 윗 상태에서의 가치는 등비급수의 합으로

$$
v_{\pi_{\texttt{left}}}(\text{top}) = \frac{1}{1 - \gamma^2}, \qquad v_{\pi_{\texttt{right}}}(\text{top}) = \frac{2\gamma}{1 - \gamma^2}
$$

가 된다. 두 값을 비교하면 $v_{\pi_{\texttt{right}}} - v_{\pi_{\texttt{left}}} = (2\gamma - 1)/(1 - \gamma^2)$이므로 부호는 $2\gamma - 1$의 부호로 결정된다.

- **$\gamma = 0$:** $v_{\pi_{\texttt{left}}} = 1$, $v_{\pi_{\texttt{right}}} = 0$이므로 **$\pi_{\texttt{left}}$가 최적**이다. 즉시 보상만 보고 결정하는 근시안적 에이전트는 첫 보상이 큰 행동을 선호한다.
- **$\gamma = 0.9$:** $v_{\pi_{\texttt{left}}} = 1/0.19 \approx 5.263$, $v_{\pi_{\texttt{right}}} = 1.8/0.19 \approx 9.474$이므로 **$\pi_{\texttt{right}}$가 최적**이다. 미래 보상을 거의 그대로 반영하는 에이전트에게는 더 큰 보상이 주기적으로 들어오는 쪽이 유리하다.
- **$\gamma = 0.5$:** $v_{\pi_{\texttt{left}}} = 1/0.75 = 4/3$, $v_{\pi_{\texttt{right}}} = 1/0.75 = 4/3$로 **두 정책의 가치가 같다**. 이 값이 정확히 손익분기점 $\gamma = 1/2$이다.

이 예에서 최적 정책은 할인 인자에 따라 달라진다.

### Exercise 3.23: 재활용 로봇의 $q_*$에 대한 벨만 방정식

상태 집합은 $\{\mathtt{h}, \mathtt{l}\}$(고/저 배터리)이고, 가능한 행동은 고배터리에서는 $\{\mathtt{s}, \mathtt{w}\}$(탐색, 대기), 저배터리에서는 $\{\mathtt{s}, \mathtt{w}, \mathtt{re}\}$(탐색, 대기, 충전)이다. 매 행동의 다음 상태는 [Exercise 3.4](#exercise-3-4)의 표를 따른다. $q_*$에 대한 벨만 최적 방정식

$$
q_*(s, a) = \sum_{s', r} p(s', r \mid s, a)\Bigl[r + \gamma \max_{a' \in \mathcal{A}(s')} q_*(s', a')\Bigr]
$$

을 다섯 개의 $(s, a)$ 쌍 각각에 대해 풀어 쓰면 다음과 같다.

$$
\begin{aligned}
q_*(\mathtt{h}, \mathtt{s}) &= \alpha\Bigl[r_{\mathtt{s}} + \gamma \max\{q_*(\mathtt{h}, \mathtt{s}),\, q_*(\mathtt{h}, \mathtt{w})\}\Bigr] \\
&\quad + (1 - \alpha)\Bigl[r_{\mathtt{s}} + \gamma \max\{q_*(\mathtt{l}, \mathtt{s}),\, q_*(\mathtt{l}, \mathtt{w}),\, q_*(\mathtt{l}, \mathtt{re})\}\Bigr], \\[4pt]
q_*(\mathtt{h}, \mathtt{w}) &= r_{\mathtt{w}} + \gamma \max\{q_*(\mathtt{h}, \mathtt{s}),\, q_*(\mathtt{h}, \mathtt{w})\}, \\[4pt]
q_*(\mathtt{l}, \mathtt{s}) &= \beta\Bigl[r_{\mathtt{s}} + \gamma \max\{q_*(\mathtt{l}, \mathtt{s}),\, q_*(\mathtt{l}, \mathtt{w}),\, q_*(\mathtt{l}, \mathtt{re})\}\Bigr] \\
&\quad + (1 - \beta)\Bigl[-3 + \gamma \max\{q_*(\mathtt{h}, \mathtt{s}),\, q_*(\mathtt{h}, \mathtt{w})\}\Bigr], \\[4pt]
q_*(\mathtt{l}, \mathtt{w}) &= r_{\mathtt{w}} + \gamma \max\{q_*(\mathtt{l}, \mathtt{s}),\, q_*(\mathtt{l}, \mathtt{w}),\, q_*(\mathtt{l}, \mathtt{re})\}, \\[4pt]
q_*(\mathtt{l}, \mathtt{re}) &= 0 + \gamma \max\{q_*(\mathtt{h}, \mathtt{s}),\, q_*(\mathtt{h}, \mathtt{w})\}.
\end{aligned}
$$

각 행에서 $\max$의 후보 집합은 다음 상태에서 가능한 행동의 집합과 일치한다. 다섯 개의 결합된 비선형 방정식을 함께 풀면 $q_*$를 얻을 수 있다.

### Exercise 3.24: 격자 세계 최선 상태 값의 정확한 계산

격자 세계 예제([예제 3.5](#example-3-5))에서 최선 상태는 특수 상태 A이다. A에서는 어떤 행동을 선택해도 보상 $+10$과 함께 A'으로 이동하고, A'에서는 통상의 격자 이동 규칙을 따른다. 최적 정책에서 A'에 도달한 뒤에는 가능한 한 빠르게 A로 돌아가야 하며, A'은 A로부터 네 칸 아래에 있으므로 위쪽으로 네 번 연속 이동하는 것이 최적이다. 이 동안의 보상은 모두 $0$이다.

따라서 A에서 출발하는 최적 보상 수열은 매 다섯 단계마다 $+10$이 한 번씩 등장하는 주기적 형태가 된다.

$$
\underbrace{10}_{R_1},\ \underbrace{0,\ 0,\ 0,\ 0}_{R_2,\ldots,R_5},\ \underbrace{10}_{R_6},\ \underbrace{0,\ 0,\ 0,\ 0}_{R_7,\ldots,R_{10}},\ \underbrace{10}_{R_{11}},\ \ldots
$$

식 (3.8)을 적용하면

$$
v_*(A) = \sum_{k=0}^{\infty} \gamma^{5k} \cdot 10 = \frac{10}{1 - \gamma^5}
$$

이 된다. $\gamma = 0.9$를 대입하면 $\gamma^5 = 0.9^5 = 0.59049$이므로

$$
v_*(A) = \frac{10}{1 - 0.59049} = \frac{10}{0.40951} = 24.4194\ldots
$$

가 되어 소수 셋째 자리까지 $v_*(A) \approx 24.419$이다. 이 값은 그림 3.5의 $24.4$와 일치한다.

<a id="exercise-3-25"></a>

### Exercise 3.25: $v_*$를 $q_*$로 표현<a id="exercise-3-25"></a>

최적 상태 가치는 그 상태에서 가능한 행동들의 최적 행동 가치 가운데 최댓값과 같다. 즉

$$
v_*(s) = \max_{a \in \mathcal{A}(s)} q_*(s, a).
$$

최적 정책은 행동 가치를 최대화하는 행동을 확률 $1$로 고르므로 이 등식이 성립한다. [Exercise 3.18](#exercise-3-18)의 정책 평균을 결정적 최댓값으로 바꿔 놓은 형태로도 읽을 수 있다.

<a id="exercise-3-26"></a>

### Exercise 3.26: $q_*$를 $v_*$와 4-인자 $p$로 표현<a id="exercise-3-26"></a>

행동 가치 $q_*(s, a)$의 분해는 정책 $\pi$에 의존하지 않으므로 [Exercise 3.19](#exercise-3-19)의 식이 그대로 적용되며, 그 안의 $v_\pi$를 $v_*$로 바꾸면 된다.

$$
q_*(s, a) = \sum_{s', r} p(s', r \mid s, a)\,\bigl[r + \gamma\, v_*(s')\bigr].
$$

이 식과 [Exercise 3.25](#exercise-3-25)의 식을 결합하면 $v_*$에 대한 벨만 최적 방정식은 다음과 같다.

$$
v_*(s) = \max_a \sum_{s', r} p(s', r \mid s, a)\bigl[r + \gamma\, v_*(s')\bigr]
$$

### Exercise 3.27: $\pi_*$를 $q_*$로 표현

최적 정책은 각 상태에서 $q_*(s, a)$를 최대화하는 행동에만 양의 확률을 부여한다. 결정적 형태로 쓰면

$$
\pi_*(a \mid s) =
\begin{cases}
1, & a \in \arg\max_{a' \in \mathcal{A}(s)} q_*(s, a'), \\
0, & \text{그 밖에}
\end{cases}
$$

이고, 일반적으로 최적 행동이 여러 개일 때는 그 안에서 임의로 분포를 정해도 최적 정책이다. 즉 $\arg\max$에 속하지 않는 행동의 확률만 $0$이면 충분하다.

### Exercise 3.28: $\pi_*$를 $v_*$와 4-인자 $p$로 표현

[Exercise 3.26](#exercise-3-26)의 식을 $q_*$에 대입하면, $q_*$를 직접 알지 못해도 $v_*$와 환경 모형 $p$만으로 최적 정책을 표현할 수 있다.

$$
\pi_*(a \mid s) =
\begin{cases}
1, & a \in \arg\max\limits_{a' \in \mathcal{A}(s)} \sum\limits_{s', r} p(s', r \mid s, a')\bigl[r + \gamma\, v_*(s')\bigr], \\
0, & \text{그 밖에.}
\end{cases}
$$

이 식은 동적 계획법에서 가치 반복(value iteration)의 수렴 후 정책을 추출하는 단계와 정확히 일치한다.

### Exercise 3.29: 두 인자 함수 $r(s, a)$와 세 인자 함수 $p(s' \mid s, a)$로 다시 쓴 벨만 방정식

세 인자 전이 확률 $p(s' \mid s, a) = \sum_r p(s', r \mid s, a)$와 두 인자 기대 보상 함수 $r(s, a) = \sum_{s', r} r \cdot p(s', r \mid s, a) = \mathbb{E}[R_{t+1} \mid S_t = s, A_t = a]$를 들여오면, 4-인자 함수 $p(s', r \mid s, a)$ 위에서 정의한 네 가지 벨만 방정식을 더 간결하게 다시 쓸 수 있다. 핵심 항등식은

$$
\sum_{s', r} p(s', r \mid s, a)\bigl[r + \gamma\, V(s')\bigr] = r(s, a) + \gamma \sum_{s'} p(s' \mid s, a)\, V(s')
$$

이며, 이를 네 개의 가치함수에 적용하면 다음과 같다.

$$
\begin{aligned}
v_\pi(s) &= \sum_{a} \pi(a \mid s)\Bigl[r(s, a) + \gamma \sum_{s'} p(s' \mid s, a)\, v_\pi(s')\Bigr], \\[4pt]
q_\pi(s, a) &= r(s, a) + \gamma \sum_{s'} p(s' \mid s, a) \sum_{a'} \pi(a' \mid s')\, q_\pi(s', a'), \\[4pt]
v_*(s) &= \max_a \Bigl[r(s, a) + \gamma \sum_{s'} p(s' \mid s, a)\, v_*(s')\Bigr], \\[4pt]
q_*(s, a) &= r(s, a) + \gamma \sum_{s'} p(s' \mid s, a) \max_{a'} q_*(s', a').
\end{aligned}
$$

이 형태는 보상에 대한 합과 다음 상태에 대한 합을 분리해 두었기 때문에 표기가 한결 간결해지며, 알고리즘 구현에서도 보상 모형과 전이 모형을 분리해 다룰 수 있다.

## 3.7 최적성과 근사

벨만 최적 방정식은 강화학습 문제의 이론적 목표를 또렷이 보여 주지만, 현실의 대규모 문제에서 정확한 최적해를 구하기는 대개 불가능하다. 가능한 상태와 행동이 너무 많고, 환경 모델을 알 수 없으며, 계산 시간과 데이터도 한정되어 있기 때문이다.

따라서 실제 강화학습에서는 **완전한 최적성**보다 제한된 자원 안에서 충분히 좋은 정책을 찾는 일이 더 중요하다. 에이전트가 자주 방문하는 상태에서 좋은 결정을 내리는 것이, 거의 방문하지 않는 상태에서 정확한 최적 행동을 아는 것보다 실용적으로 더 값질 수 있다. 이 관점에서 함수 근사, 샘플 기반 학습, 온라인 학습의 필요성이 자연스럽게 떠오른다.

또한 최적성은 항상 모델링 선택에 상대적이다. 어떤 상태 표현을 쓰는지, 어떤 행동을 허용하는지, 보상을 어떻게 정의하는지에 따라 최적 정책 자체가 달라진다. 그러므로 MDP는 현실을 완전히 복제한 것이 아니라 의사결정 문제를 다루기 위한 추상화이며, 그 추상화 안에서의 최적성을 목표로 삼는다.

## 3.8 요약

3장은 강화학습 문제를 엄밀하게 기술하기 위한 기본 언어를 제공한다. 에이전트와 환경은 상태, 행동, 보상을 주고받으며 상호작용하고, 현재 상태와 행동만으로 다음 상태와 보상의 분포를 정할 수 있을 때 이 관계를 MDP로 표현한다.

이 장의 핵심 개념은 다음과 같이 정리할 수 있다.

- **보상**은 에이전트의 목표를 정의하는 스칼라 신호이다.
- **이득**은 에이전트가 장기적으로 최대화하려는 누적 보상이다.
- **정책**은 상태에서 행동으로의 선택 규칙이다.
- **가치 함수**는 특정 정책을 따랐을 때 상태나 행동이 갖는 장기적 기대 가치를 나타낸다.
- **벨만 방정식**은 가치를 즉시 보상과 다음 상태 가치의 합으로 재귀적으로 정의한다.
- **최적 가치 함수**와 **최적 정책**은 가능한 정책들 가운데 장기 이득을 최대화하는 기준을 제공한다.

결국 3장은 이후 모든 강화학습 알고리즘의 공통 기반을 세운다. 2장의 밴딧 문제가 상태가 하나뿐인 특수한 경우였다면, 3장의 MDP는 상태 전이와 지연된 보상이 있는 일반적 순차 의사결정 문제를 다루는 형식적 틀이다.